# Highway Networks on CIFAR-10

A small PyTorch implementation of
[Highway Networks](https://arxiv.org/abs/1505.00387) by Srivastava, Greff and Schmidhuber (2015),
trained on CIFAR-10.

The notebook follows the paper in three places:

1. the highway layer is Equation (3) of the paper;
2. the first layer is a plain layer that changes the dimensionality, followed by highway layers
   (Section 2.1);
3. the transform-gate bias starts negative, so the network is biased towards carrying its input
   at the beginning of training (Section 2.2).

The paper's CIFAR-10 models are convolutional. To keep the code short and readable, this notebook
flattens the images and uses fully connected layers. The highway mechanism is exactly the same.

A plain network with the same depth is trained as a baseline. Its width is chosen so that both
models have almost the same number of parameters, as in the paper.

**Running it.** A GPU is recommended but not required. On Colab choose
*Runtime > Change runtime type > GPU*; `torch` and `torchvision` are already installed there.
Locally, run `uv sync` first and select the project kernel. CIFAR-10 is downloaded to `./data`
on the first run.

## The highway layer

A plain layer computes $y = H(x, W_H)$. A highway layer adds two gates:

$$
y = H(x, W_H) \odot T(x, W_T) + x \odot C(x, W_C),
\qquad C(x, W_C) = 1 - T(x, W_T).
$$

The paper uses a sigmoid transform gate,

$$
T(x) = \sigma(W_T^\top x + b_T),
$$

so every unit has its own gate value between 0 and 1. The layer therefore interpolates between two
behaviours:

$$
y =
\begin{cases}
x, & \text{if } T(x) = 0 \quad \text{(carry the input)},\\
H(x, W_H), & \text{if } T(x) = 1 \quad \text{(transform the input)}.
\end{cases}
$$

The same is true for the Jacobian: when $T(x)=0$ the layer's Jacobian is the identity, so gradients
can travel through it unchanged.

For Equation (3) to work, $x$, $y$, $H(x)$ and $T(x)$ must have the same shape. This is why the
paper starts with a plain layer that changes the dimensionality and then stacks highway layers.

**The important initialization detail.** The bias $b_T$ starts at a negative value (the paper
uses values such as $-1$ and $-3$; we use $-2$). Since $\sigma(-2) \approx 0.12$, the gates start
mostly closed and the deep network initially behaves like a shallow one. The gates then open up
during training where they are useful.

## Setup

In [ ]:
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    print(torch.cuda.get_device_name(0))

In [ ]:
SEED = 0
DATA_ROOT = Path("data")
BATCH_SIZE = 256
EPOCHS = 15
LR = 1e-3
DEPTH = 12
HIGHWAY_WIDTH = 512
PLAIN_WIDTH = 692  # wider, so both models have about the same number of parameters
GATE_BIAS = -2.0
NUM_WORKERS = 2  # set to 0 on Windows if the data loader hangs


def set_seed(seed=SEED):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

## Data

CIFAR-10 has 50,000 training and 10,000 test images of size $32 \times 32$, in 10 classes.
Training images get random crops and horizontal flips; both splits are normalized with the
dataset mean and standard deviation.

In [ ]:
MEAN = (0.4914, 0.4822, 0.4465)
STD = (0.2470, 0.2435, 0.2616)

train_tf = transforms.Compose(
    [
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD),
    ]
)
test_tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

train_set = datasets.CIFAR10(DATA_ROOT, train=True, download=True, transform=train_tf)
test_set = datasets.CIFAR10(DATA_ROOT, train=False, download=True, transform=test_tf)
classes = test_set.classes

train_loader = DataLoader(
    train_set,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=device.type == "cuda",
)
test_loader = DataLoader(
    test_set,
    batch_size=512,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=device.type == "cuda",
)

# A fixed batch used later to inspect the gates.
probe_x, _ = next(iter(test_loader))
probe_x = probe_x[:256].to(device)

print(f"{len(train_set):,} train / {len(test_set):,} test images")

## The highway layer in code

The code is a direct translation of Equation (3): one linear layer produces $H$, one produces the
gate logits, and the sigmoid turns those into $T$. The full gate tensor is stored so we can inspect
it after training.

In [ ]:
class Highway(nn.Module):
    def __init__(self, dim: int, gate_bias: float = GATE_BIAS):
        super().__init__()
        self.H = nn.Linear(dim, dim)
        self.gate = nn.Linear(dim, dim)
        nn.init.constant_(self.gate.bias, gate_bias)  # start biased towards carry
        self.last_gate = torch.zeros(dim)  # filled on every forward pass

    def forward(self, x):
        h = F.relu(self.H(x))
        t = torch.sigmoid(self.gate(x))
        self.last_gate = t.detach()
        return h * t + x * (1 - t)

## Plain and highway networks

Both models have the paper's overall shape:

* a plain first layer that changes the dimensionality (`Flatten -> Linear -> ReLU`),
* `DEPTH` hidden blocks,
* a linear classifier.

Only the hidden blocks differ. A highway block has two linear maps, so the plain model is made
wider (692 instead of 512 units) to keep the parameter counts comparable. The paper does the same
with widths 71 and 50.

In [ ]:
class Net(nn.Module):
    def __init__(self, highway: bool, width: int, depth: int = DEPTH):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Flatten(),
            nn.Linear(3 * 32 * 32, width),
            nn.ReLU(),
        )
        if highway:
            self.blocks = nn.ModuleList(Highway(width) for _ in range(depth))
        else:
            self.blocks = nn.ModuleList(
                nn.Sequential(nn.Linear(width, width), nn.ReLU()) for _ in range(depth)
            )
        self.head = nn.Linear(width, 10)

    def forward(self, x):
        x = self.stem(x)
        for block in self.blocks:
            x = block(x)
        return self.head(x)


def n_params(model):
    return sum(p.numel() for p in model.parameters())

## Training

A standard classification loop: Adam, cross-entropy loss, and a cosine learning-rate schedule.
For the highway model we also record the mean gate value of every layer after each epoch.

In [ ]:
@torch.no_grad()
def accuracy(model, loader):
    model.eval()
    correct = total = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        correct += (model(x).argmax(dim=1) == y).sum().item()
        total += y.numel()
    return correct / total


@torch.no_grad()
def gate_means_on_batch(model, x):
    model.eval()
    model(x)
    return [block.last_gate.mean().item() for block in model.blocks]


def train(model, track_gates: bool = False, epochs: int = EPOCHS, lr: float = LR):
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    loss_fn = nn.CrossEntropyLoss()

    history = {"loss": [], "acc": [], "gates": []}
    start = time.time()
    for epoch in range(1, epochs + 1):
        model.train()
        running = 0.0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad(set_to_none=True)
            loss = loss_fn(model(x), y)
            loss.backward()
            opt.step()
            running += loss.item() * y.numel()
        sched.step()

        history["loss"].append(running / len(train_set))
        history["acc"].append(accuracy(model, test_loader))
        if track_gates:
            history["gates"].append(gate_means_on_batch(model, probe_x))
        print(
            f"epoch {epoch:2d}/{epochs} | loss {history['loss'][-1]:.3f} "
            f"| test acc {history['acc'][-1]:.1%}"
        )

    print(f"trained in {time.time() - start:.1f}s")
    return history

## Train the baseline

In [ ]:
set_seed()
plain = Net(highway=False, width=PLAIN_WIDTH)
print(f"plain:   {n_params(plain):,} parameters")
plain_hist = train(plain)

## Train the highway network

In [ ]:
set_seed()
highway = Net(highway=True, width=HIGHWAY_WIDTH)
print(f"highway: {n_params(highway):,} parameters")
highway_hist = train(highway, track_gates=True)

## Results

Two identical training setups, with and without the gates.

In [ ]:
fig, (ax_loss, ax_acc) = plt.subplots(1, 2, figsize=(10, 3.5))

for name, hist in [("plain", plain_hist), ("highway", highway_hist)]:
    ax_loss.plot(hist["loss"], label=name)
    ax_acc.plot([100 * a for a in hist["acc"]], label=name)

ax_loss.set(xlabel="epoch", ylabel="train loss", title="Training loss")
ax_acc.set(xlabel="epoch", ylabel="test accuracy (%)", title="Test accuracy")
for ax in (ax_loss, ax_acc):
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

for name, hist in [("plain", plain_hist), ("highway", highway_hist)]:
    print(f"{name:8s} final test accuracy: {hist['acc'][-1]:.1%}")

## How the gates change during training

Each row is one highway layer and each column is an epoch. Bright cells mean the layer transforms
its input ($T$ close to 1); dark cells mean it mostly carries the input forward ($T$ close to 0).
The gates start near $\sigma(-2) \approx 0.12$ and move during training.

In [ ]:
gate_history = torch.tensor(highway_hist["gates"]).T  # layers x epochs

plt.figure(figsize=(8, 4))
plt.imshow(gate_history, aspect="auto", origin="lower", cmap="viridis", vmin=0, vmax=1)
plt.colorbar(label="mean transform gate T")
plt.xlabel("epoch")
plt.ylabel("highway layer")
plt.title("Mean gate value of every layer during training")
plt.tight_layout()
plt.show()

## What a single image does to the gates

The paper's Figure 2 inspects the gates of a trained network. Here is the same idea: one test image,
all layers, and all 512 gate values per layer. Many units are strongly closed or strongly open, so
the network uses its gates selectively instead of transforming everything.

In [ ]:
image, label = test_set[0]
with torch.no_grad():
    highway(image.unsqueeze(0).to(device))
gates = []
for block in highway.blocks:
    assert isinstance(block, Highway)
    gates.append(block.last_gate[0].cpu())
gates = torch.stack(gates)

fig, (ax_map, ax_hist) = plt.subplots(1, 2, figsize=(12, 4), gridspec_kw={"width_ratios": [3, 1]})
im = ax_map.imshow(gates, aspect="auto", cmap="viridis", vmin=0, vmax=1)
ax_map.set(
    xlabel="hidden unit",
    ylabel="layer",
    title=f"Transform gates for one image ({classes[label]})",
)
fig.colorbar(im, ax=ax_map, label="T")

ax_hist.hist(gates.flatten(), bins=50, range=(0, 1), color="tab:orange")
ax_hist.set(xlabel="T", ylabel="units", title="Gate distribution")
plt.tight_layout()
plt.show()

## The information highway

A highway layer can pass its input through almost unchanged. The plot below measures how similar
each layer's output is to its input using cosine similarity. Values close to 1 mean the layer
copied its input; values near 0 mean it changed the representation substantially.

In [ ]:
@torch.no_grad()
def layer_similarity(model, x):
    model.eval()
    h = model.stem(x)
    sims = []
    for block in model.blocks:
        h_new = block(h)
        sims.append(F.cosine_similarity(h_new, h, dim=1).mean().item())
        h = h_new
    return sims


plt.figure(figsize=(8, 3.5))
plt.plot(range(1, DEPTH + 1), layer_similarity(highway, probe_x), "o-", label="highway")
plt.plot(range(1, DEPTH + 1), layer_similarity(plain, probe_x), "o-", label="plain")
plt.axhline(0, color="gray", linewidth=1)
plt.ylim(-0.2, 1)
plt.xlabel("layer")
plt.ylabel("cosine similarity(output, input)")
plt.title("How much each layer changes its input")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## A look at predictions

In [ ]:
images, labels = next(iter(test_loader))
images, labels = images[:8], labels[:8]  # one batch holds 512, show 8
with torch.no_grad():
    probs = highway(images.to(device)).softmax(dim=1).cpu()
labels = labels.tolist()
preds = probs.argmax(dim=1).tolist()
confs = probs.max(dim=1).values.tolist()

fig, axes = plt.subplots(2, 4, figsize=(10, 5.5))
for ax, img, label, pred, conf in zip(axes.flat, images, labels, preds, confs, strict=True):
    rgb = (img * torch.tensor(STD)[:, None, None] + torch.tensor(MEAN)[:, None, None]).clamp(0, 1)
    ax.imshow(rgb.permute(1, 2, 0))
    ax.set_title(f"true: {classes[label]}\npred: {classes[pred]} ({conf:.0%})", fontsize=9)
    ax.axis("off")

plt.tight_layout()
plt.show()